# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** Mathematical Modeling & Quantum Mechanics

---
*Nota Institucional: Este notebook funge como un solucionador numérico para Ecuaciones Diferenciales Parciales (PDEs). Aplica el método de Crank-Nicolson sobre la ecuación de Schrödinger dependiente del tiempo para modelar el túnel cuántico. Operacionalmente, abstrae el fenómeno físico en la inversión recurrente de matrices tridiagonales dispersas sobre una malla espacial discretizada, propagando funciones de onda gaussianas.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sparse
import scipy.sparse.linalg as spla
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')

# Constantes Cuánticas Simplificadas (hbar = 2m = 1)
dt = 0.005
time_steps = 400

### 1. Ingestión del Espacio y la Barrera de Potencial $V(x)$
Cargamos la malla espacial y el perfil de la barrera pre-calculado, definiendo la topología unidimensional sobre la cual interactuará nuestra partícula (paquete de ondas).

In [ ]:
df_pot = pd.read_csv('../data/quantum_barrier_profile.csv')
x = df_pot['x'].values
V = df_pot['V'].values

N = len(x)
dx = x[1] - x[0]

print(f"[*] Discretización de malla: {N} nodos espaciales. Diferencial dx = {dx:.4f}")

### 2. Condición Inicial: El Paquete de Ondas Gaussiano $\Psi(x,0)$
Inicializamos la partícula como una densidad de probabilidad inyectándole momento intrínseco ($k_0$) apuntando hacia la barrera limitante.

In [ ]:
x0 = -2.0      # Posición inicial
sigma = 0.25   # Dispersión (Incertidumbre espacial)
k0 = 15.0      # Momento (Velocidad de grupo)

# Construcción de Campo Complejo
psi_0 = np.exp(-0.5 * ((x - x0)/sigma)**2) * np.exp(1j * k0 * x)

# Normalización Unitaria (La suma de probabilidades debe ser exactamente 1)
norm = np.sum(np.abs(psi_0)**2) * dx
psi = psi_0 / np.sqrt(norm)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, np.abs(psi)**2, color='cyan', label='Densidad Inicial $|\\Psi(x,0)|^2$')
ax.fill_between(x, V / np.max(V) * np.max(np.abs(psi)**2), color='red', alpha=0.3, label='Barrera de Potencial V(x)')
ax.set_title("Condiciones de Frontera y Paquete Inicial", fontsize=14, pad=10)
ax.legend(loc="upper left")
plt.show()

### 3. Matrices Dispersas Tridiagonales (Crank-Nicolson)
Al discretizar la Ecuación de Schrödinger:
$i\frac{\partial \Psi}{\partial t} = \left(-\frac{\partial^2}{\partial x^2} + V(x)\right)\Psi$

Obtenemos el operador del sistema hamiltoniano $(H)$. El método incondicionalmente estable de Crank-Nicolson se formaliza aislando las iteraciones del tiempo futuro $\Psi^{n+1}$:
$\left(I + \frac{i \Delta t}{2} H \right) \Psi^{n+1} = \left(I - \frac{i \Delta t}{2} H \right) \Psi^{n}$

In [ ]:
# Coeficiente Laplaciano
alpha = 1.0 / (dx**2)

# Diagonal Principal: (2/dx^2 + V(x))
d0 = 2.0 * alpha + V
# Diagonales Secundarias: (-1/dx^2)
d1 = -alpha * np.ones(N)
dm1 = -alpha * np.ones(N)

# Construir matriz dispersa altamente eficiente (SciPy sparse)
H = sparse.diags([dm1, d0, d1], offsets=[-1, 0, 1], shape=(N, N), format='csc')

# Construir Matrices de Evolución Temporal (M_L * Psi_new = M_R * Psi_old)
I = sparse.eye(N, format='csc')
coef = (1j * dt) / 2.0

M_L = I + coef * H
M_R = I - coef * H

# Factorización y Solucionador Lineal Directo
solver = spla.factorized(M_L)
print("[*] Matriz Vectorial Factorizada. Solucionador directo lineal inicializado.")

### 4. Propagación en el Tiempo y Cálculo del Coeficiente de Transmisión
Avanzamos bucle por bucle resolviendo $A x = b$. La parte de la onda que consiga atravesar el ancho numérico de $V_0=50$ determinará experimentalmente la ocurrencia del Fenómeno de Túnel Cuántico.

In [ ]:
historico_densidades = []

# Ciclo de propogación Crank-Nicolson
for step in range(time_steps):
    # Multiplicación mat-vec para el termino independiente b
    b = M_R.dot(psi)
    # Inversión matemática para resolver Psi^{n+1}
    psi = solver(b)
    
    if step % 80 == 0 or step == time_steps - 1:
        historico_densidades.append(np.abs(psi)**2)
        
# Cuantificación de Integrales
# Densidad total de probabilidad debe seguir siendo 1 (conservación)
prob_total = np.sum(np.abs(psi)**2) * dx

# Probabilidad Transmitida (Integrando el espacio después de X=1.0)
idx_trans = np.where(x > 1.0)[0]
prob_trans = np.sum(np.abs(psi[idx_trans])**2) * dx

print(f"[+] Conservación del Paquete (Norma): {prob_total:.4f}")
print(f"[+] Coeficiente de Transmisión Cuántica (Probabilidad de Tunelamiento): {prob_trans*100:.3f} %")

### Visualización Final
Mostramos cómo la matriz dispersa tridiagonal ha "cortado" y "rebotado" el frente de onda matemático. Una fracción estadísticamente baja del vector logró atravesar la masa sólida debido al decaimiento exponencial interno en la matriz resolutora.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.fill_between(x, V / np.max(V) * np.max(historico_densidades[0]), color='red', alpha=0.1, label='Matriz del Obstáculo (V_x)')

colors = ['cyan', 'dodgerblue', 'blueviolet', 'magenta', 'lime', 'yellow']

for i, p_dens in enumerate(historico_densidades):
    ax.plot(x, p_dens, color=colors[i % len(colors)], label=f'Paso de Tiempo {i*80}')

ax.set_title("Túnel Cuántico: Dispersión y Transmisión de la Señal de Densidad", fontsize=15, pad=15)
ax.set_xlabel("Malla Espacial X")
ax.set_ylabel("Densidad de Probabilidad $|\\Psi(x,t)|^2$")
ax.set_xlim(-4, 4)
ax.legend(loc="upper right", fontsize=9)
plt.grid(alpha=0.2)
plt.show()